In [1]:
# Carico la cartella one drive in cui ci sono:
# cartella video nutrition estimation / video annotation nutrition estimation
# cartella video nutrition change / video annotation nutrition change
# cartella video image nutrition estimation/ video annotation image estimation

# come ho scaricato i video?
# partendo dai file json ho lanciato lo script downloader sugli id estratti
# dei video scaricati ho fatto un samplig a N=128 FRAME
# caricato su one drive ed adesso linko da questo notebook
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Verifico scheda video e versione compilatore NVCC disponibile nell'ambiente**
- in questo caso A100 con 80Gb di VRAM

In [2]:
!nvidia-smi
!nvcc --version

Sat Sep 12 10:06:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             54W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:

# installo PyTorch 2.7.0
!pip install torch==2.7.0 torchvision==0.22.0 --index-url https://download.pytorch.org/whl/cu128


Looking in indexes: https://download.pytorch.org/whl/cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 202.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 244.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 260.9 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 726.9/726.9 MB 72.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 609.6/609.6 MB 64.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 134.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 199.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.4/260.4 MB 117.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.1/292.1 MB 80.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 MB 148.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [4]:

# Rimuovo la vecchia copia (che ha patch residue) e ri-clono il repo pulito
import os, shutil
if os.path.exists("/content/VideoLLaMA2"):
    shutil.rmtree("/content/VideoLLaMA2")
!git clone https://github.com/DAMO-NLP-SG/VideoLLaMA2.git /content/VideoLLaMA2   #clono il repo pulito

import sys
if "/content/VideoLLaMA2" not in sys.path:
    sys.path.insert(0, "/content/VideoLLaMA2")


Cloning into '/content/VideoLLaMA2'...
remote: Enumerating objects: 986, done.
remote: Counting objects: 100% (433/433), done.
remote: Compressing objects: 100% (187/187), done.
remote: Total 986 (delta 365), reused 246 (delta 246), pack-reused 553 (from 2)
Receiving objects: 100% (986/986), 55.81 MiB | 18.92 MiB/s, done.
Resolving deltas: 100% (660/660), done.


In [5]:
%cd VideoLLaMA2/

/content/VideoLLaMA2


In [6]:
!pip install --upgrade pip setuptools wheel #installo librerie utilitarie per l'utilizzo di librerie pre-compilate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 84.9 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.47.0
    Uninstalling wheel-0.47.0:
      Successfully uninstalled wheel-0.47.0
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [7]:
# Versioni ufficiali richieste da VideoLLaMA2 (vedi pyproject.toml del repo):
#   transformers==4.40.0, tokenizers==0.19.1, accelerate==0.26.1, decord==0.6.0
# NB: transformers 4.38.2 era troppo vecchio per le classi custom Videollama2*ForCausalLM.
!pip install "transformers==4.40.0" "tokenizers==0.19.1" "accelerate==0.26.1" \
             peft timm einops "decord==0.6.0" av opencv-python


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 196.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 174.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 264.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 256.2 MB/s  0:00:00
  Attempting uninstall: huggingface-hub━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/6 [av]
    Found existing installation: huggingface_hub 1.23.0━━━━━━━ 1/6 [av]
    Uninstalling huggingface_hub-1.23.0:━━━━━━━━━━━━━━━━━━━━━━ 1/6 [av]
      Successfully uninstalled huggingface_hub-1.23.0━━━━━━━━━━━━━ 2/6 [huggingface-hub]
  Attempting uninstall: tokenizersm━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/6 [huggingface-hub]
    Found existing installation: tokenizers 0.22.2━━━━━━━━━━━━ 2/6 [huggingface-hub]
    Uninstalling tokenizers-0.22.2:m━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/6 [huggingface-hub]
      Successfully uninstalled tokenizers-0.22.2━━━━━━━

In [8]:
# NON esiste una env var "TRANSFORMERS_ATTN_IMPLEMENTATION" in transformers.
# L'attention eager va richiesta passando attn_implementation="eager" a model_init
print("Eager attention verra' impostato via attn_implementation='eager' in model_init")


Eager attention verra' impostato via attn_implementation='eager' in model_init


In [9]:
import os
# Evita il lookup del secret HF_TOKEN di Colab che va in timeout e "appende" il download
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"   # download piu' stabile
os.environ.pop("HF_TOKEN", None)

import gc, sys, torch
import pathlib

# Libero la VRAM da tentativi precedenti
for name in ['model', 'processor', 'tokenizer']:
    if name in dir():
        exec(f"del {name}")
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f"VRAM libera: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GiB")

# -----------------------------------------------------------------------------
# PATCH encoder.py: il repo forza a mano config._attn_implementation="flash_attention_2"
# sulla CLIP/Siglip vision tower, ma questi modelli non supportano flash-attn -> ValueError.
# Sostituiamo con "eager" DIRETTAMENTE nel file sorgente, prima di importare videollama2.
# -----------------------------------------------------------------------------
_enc = pathlib.Path("/content/VideoLLaMA2/videollama2/model/encoder.py")
_txt = _enc.read_text()
if '"flash_attention_2"' in _txt:
    _enc.write_text(_txt.replace('"flash_attention_2"', '"eager"'))
    print("Patch applicata: flash_attention_2 -> eager in encoder.py")#utilizziamo eager come attention
else:
    print("Nessuna patch necessaria (flash_attention_2 non trovato in encoder.py)")

# Svuoto la cache dei moduli gia' importati, cosi' l'import qui sotto rilegge il file patchato
for _m in list(sys.modules):
    if _m.startswith("videollama2"):
        del sys.modules[_m]

# 1) Scarico lo snaspshot VIDEOLLAMA2-7B
from huggingface_hub import snapshot_download
model_path = snapshot_download("DAMO-NLP-SG/VideoLLaMA2-7B", resume_download=True)
print("Modello scaricato in:", model_path)

# 2) Inizializzo passando il path LOCALE (niente altre chiamate di rete).
from videollama2 import model_init as videollama2_model_init

device = "cuda:0"
print("Inizio caricamento modello...")
model, processor, tokenizer = videollama2_model_init(#carico modello
    model_path,
    device_map={"": device},
    attn_implementation="eager",   # forza eager sul modello linguistico
    use_flash_attn=False,
)


#imposto il modello in modalità Inferenza
model.eval()
print("Modello VideoLLaMA2-7B caricato con successo — dtype:", next(model.parameters()).dtype)
print("num_frames usati dal processor:", getattr(model.config, "num_frames", "default (8)"))


VRAM libera: 78.83 GiB
Patch applicata: flash_attention_2 -> eager in encoder.py


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Modello VideoLLaMA2-7B caricato con successo — dtype: torch.float16
num_frames usati dal processor: default (8)


---
# Parte 1 — Riproduzione della baseline HD-EPIC (VideoLLaMA 2 7B)

Obiettivo: verificare il valore riportato nel paper HD-EPIC (Tab. 2, colonna **Nutrition**)
per VideoLLaMA 2 7B, cioe' **32.7%**, contro un baseline random del 20%.

 **L'accuracy di categoria e' la media delle accuracy per prototipo**, non l'accuracy
   aggregata sulle domande. 

## 1.1 Motore di valutazione

Prompt, parsing delle risposte e metriche stanno in un unico modulo, condiviso con il
notebook di LLaVA-Video. E' questo che rende confrontabili i due modelli: se ogni
notebook avesse il suo parser, le differenze di accuracy misurerebbero anche le
differenze fra i parser.

La cella scrive il modulo su disco e lo importa. La copia di riferimento sta in
`inference_folder/hdepic_eval.py` nel progetto.

In [10]:
%%writefile /content/hdepic_eval.py
"""
hdepic_eval - motore di valutazione per HD-EPIC VQA, categoria Nutrition.

Usato da entrambi i notebook di inferenza (VideoLLaMA2 e LLaVA-Video): tenere
prompt, parsing e metriche in un unico posto e' cio' che rende confrontabili i
risultati dei due modelli.

La parte specifica del modello e' tutta in `infer_fn(visual, prompt) -> str`.

Protocollo (fedele al paper HD-EPIC):
  - domande a 5 scelte, risposta come LETTERA (A-E)
  - accuracy di categoria = MEDIA DELLE ACCURACY PER PROTOTIPO
  - le domande con file mancanti o corrotti sono ESCLUSE, non contate sbagliate
"""
import json, os, re, glob, unicodedata
import pandas as pd

VERSION = "1.0"
LETTERS = "ABCDEFGH"

# Valori di riferimento del paper HD-EPIC (Tab. 2, colonna Nutrition, % accuracy)
PAPER_NUTRITION = {
    "VideoLLaMA 2 7B":       32.7,
    "LLaVA-Video 7B":        38.7,
    "Llama 3.2 90B (blind)": 36.7,
    "Gemini Pro":            34.7,
    "Human":                 85.0,
    "Random":                20.0,
}

# task -> (file di annotation, modalita' di input nativa)
TASK_FILES = {
    "video_nutrition_estimation": ("nutrition_video_nutrition_estimation.json", "video"),
    "nutrition_change":           ("nutrition_nutrition_change.json",           "video"),
    "image_nutrition_estimation": ("nutrition_image_nutrition_estimation.json", "images"),
}
VIDEO_TASKS = [t for t, (_, m) in TASK_FILES.items() if m == "video"]


# costruisco indici dei file
def build_video_index(roots):
    """Indicizza TUTTI gli mp4 sotto `roots` -> {'sampled': {id: path}, 'full': {id: path}}.

    Indicizzare globalmente invece che cartella-per-task e' importante: lo stesso
    video puo' servire a piu' task ed essere stato scaricato in una sola cartella.
    """
    idx = {"sampled": {}, "full": {}}
    for root in roots:
        for p in glob.glob(os.path.join(root, "**", "*.mp4"), recursive=True):
            b = os.path.basename(p)
            if b.endswith("_sampled.mp4"):
                idx["sampled"].setdefault(b[:-len("_sampled.mp4")], p)
            else:
                idx["full"].setdefault(b[:-len(".mp4")], p)
    return idx

#indicizzo i pannelli (serve per provare i modelli con l'approccio Panels)
def build_panels_index(roots):
    """Indicizza le cartelle di pannelli -> {video_id: [path dei panel ordinati]}.

    Struttura attesa (output di mosaic.py): <root>/<video_id>/<video_id>_panel_0000.jpg
    """
    idx = {}
    for root in roots:
        if not os.path.isdir(root):
            continue
        for d in sorted(os.listdir(root)):
            full = os.path.join(root, d)
            if not os.path.isdir(full):
                continue
            panels = sorted(glob.glob(os.path.join(full, "*.jpg")) +
                            glob.glob(os.path.join(full, "*.jpeg")) +
                            glob.glob(os.path.join(full, "*.png")))
            if panels:
                idx.setdefault(d, panels)
    return idx

#per leggere il video dal Path
def resolve_video(video_id, index, prefer="sampled"):
    """-> (path, sorgente) con sorgente in {'sampled','full',None}."""
    order = ("sampled", "full") if prefer == "sampled" else ("full", "sampled")
    for src in order:
        if video_id in index[src]:
            return index[src][video_id], src
    return None, None

#ora minuto secondi -> secondi
def hhmmss_to_sec(t):
    """'00:13:43.393' -> 823.393"""
    h, m, s = t.split(":")
    return int(h) * 3600 + int(m) * 60 + float(s)

#carico le annotazioni
def load_annotations(ann_dir, task):
    with open(os.path.join(ann_dir, TASK_FILES[task][0])) as f:
        return json.load(f)


#  prompt verso il vlm che deve rispondere
ANSWER_INSTRUCTION = "Answer with the option's letter from the given choices directly."

# 
SYSTEM_PROMPT = (
    "You are a precise video analysis assistant. Track ingredients, actions and "
    "weight measurements on digital scales to answer nutrition-related questions. "
    "Base your answer ONLY on the visual evidence provided."
)


def build_prompt(question, choices, system=None):
    """Prompt a lettere A-E. `system=None` = versione minimale, fedele al paper."""
    opts = "\n".join(f"{LETTERS[i]}. {c}" for i, c in enumerate(choices))
    head = f"{system}\n\n" if system else ""
    return f"{head}{question}\n{opts}\n{ANSWER_INSTRUCTION}"


NUTRIENT_RE = re.compile(r"\b(?:highest|higher)\s+(calories|carbs|fat|protein)\b", re.I)


def q_type_of(task, question):
    """Tipo di domanda: il nutriente chiesto, oppure il nome del prototipo."""
    m = NUTRIENT_RE.search(question)
    if m:
        return m.group(1).lower()
    return "nutrition_change" if task == "nutrition_change" else "unknown"


# parsing
def _norm(s):
    s = unicodedata.normalize("NFKD", s).lower()
    return re.sub(r"[^a-z0-9]+", " ", s).strip()

# parso la risposta del modello per estrarre la risposta (per sicurezza cone le regex cerco di cogliere tutte le possibilità)
def parse_answer(response, choices):
    """Estrae l'indice della scelta -> (idx, metodo).

    A livelli, dal piu' stretto al piu' permissivo. Il metodo viene salvato nei
    risultati, cosi' e' sempre ispezionabile COME si e' arrivati a quell'indice.
    idx = -1 significa 'risposta non interpretabile'.
    """
    n = len(choices)
    valid = LETTERS[:n]
    s = (response or "").strip()
    if not s:
        return -1, "empty"

    # 1) inizia con la lettera: "C", "C.", "(C)", "C) spring greens"
    #    il lookahead evita di leggere la C di "Calories changed by..."
    m = re.match(r"^\W*\(?([A-Za-z])\)?(?=[\s.,:;)\-]|$)", s)
    if m and m.group(1).upper() in valid:
        return valid.index(m.group(1).upper()), "leading_letter"

    # 2) formula esplicita: "the answer is C", "Option B"
    m = re.search(r"\b(?:answer|option|choice)\s*(?:is|:)?\s*\(?([A-Za-z])\)?\b", s, re.I)
    if m and m.group(1).upper() in valid:
        return valid.index(m.group(1).upper()), "answer_is_letter"

    # 3) ha ricopiato il testo di una scelta invece della lettera
    ns = _norm(s)
    hits = [i for i, c in enumerate(choices) if _norm(c) and _norm(c) in ns]
    if len(hits) == 1:
        return hits[0], "choice_text"
    hits = [i for i, c in enumerate(choices) if ns and ns in _norm(c)]
    if len(hits) == 1:
        return hits[0], "choice_text_partial"

    # 4) una sola lettera valida isolata in tutto il testo
    cand = {x.upper() for x in re.findall(r"\b([A-Za-z])\b", s)} & set(valid)
    if len(cand) == 1:
        return valid.index(cand.pop()), "single_letter_in_text"

    # 5) ha risposto con una cifra nonostante il formato a lettere
    m = re.match(r"^\W*(\d)\b", s)
    if m and 1 <= int(m.group(1)) <= n:
        return int(m.group(1)) - 1, "digit_1indexed"

    return -1, "unparsed"


# integrita' dati / pre-volo
def is_readable(path, cache=None):
    """Il file esiste ED e' decodificabile?
     Contarli come risposte sbagliate
    falserebbe l'accuracy, quindi vanno individuati PRIMA di girare.
    """
    if cache is not None and path in cache:
        return cache[path]
    ok = False
    if path and os.path.exists(path) and os.path.getsize(path) > 0:
        try:
            import cv2
            cap = cv2.VideoCapture(path)
            ok = bool(cap.read()[0])
            cap.release()
        except Exception:
            ok = False
    if cache is not None:
        cache[path] = ok
    return ok

# cerca di capire che input si aspetta il modello (video, immagini o pannelli)
def _required_inputs(item, task, index, panels_index, input_mode, prefer):
    """-> [(video_id, risorsa)] necessaria a rispondere alla domanda.

    `risorsa` e' un path mp4, una lista di path di pannelli, o None se assente.
    """
    modality = TASK_FILES[task][1]
    out = []
    if modality == "images":
        # 5 fotogrammi a timestamp precisi: serve per forza il video integrale
        for key in sorted(item["inputs"], key=lambda k: int(re.search(r"\d+", k).group())):
            inp = item["inputs"][key]
            path, _ = resolve_video(inp["id"], index, prefer="full")
            out.append((inp["id"], path))
    elif input_mode == "panels":
        for inp in item["inputs"].values():
            out.append((inp["id"], (panels_index or {}).get(inp["id"])))
    else:
        for inp in item["inputs"].values():
            path, _ = resolve_video(inp["id"], index, prefer=prefer)
            out.append((inp["id"], path))
    return out


def usable_items(ann_dir, task, index, panels_index=None, input_mode="video",
                 prefer="sampled", cache=None):
    """Divide le domande in (utilizzabili, scartate-con-motivo)."""
    ann = load_annotations(ann_dir, task)
    ok, skipped = {}, {}
    for key, item in ann.items():
        bad = []
        for vid, res in _required_inputs(item, task, index, panels_index,
                                         input_mode, prefer):
            if isinstance(res, list):          # pannelli: ne basta almeno uno
                if not res:
                    bad.append(vid)
            elif not is_readable(res, cache):  # video: deve aprirsi davvero
                bad.append(vid)
        if bad:
            skipped[key] = sorted(set(bad))
        else:
            ok[key] = item
    return ok, skipped


def preflight(ann_dir, index, tasks=None, panels_index=None, input_mode="video",
              prefer="sampled", cache=None):
    """Tabella di copertura reale. Da stampare SEMPRE prima di un run."""
    cache = {} if cache is None else cache
    rows, details = [], {}
    for task in (tasks or list(TASK_FILES)):
        ann = load_annotations(ann_dir, task)
        ok, skipped = usable_items(ann_dir, task, index, panels_index,
                                   input_mode, prefer, cache)
        details[task] = skipped
        rows.append({"task": task,
                     "input": "frames" if TASK_FILES[task][1] == "images" else input_mode,
                     "domande": len(ann),
                     "utilizzabili": len(ok),
                     "scartate": len(skipped),
                     "file_inutilizzabili": len({v for ids in skipped.values() for v in ids})})
    df = pd.DataFrame(rows)
    df.loc[len(df)] = {"task": "TOTALE", "input": "", "domande": df["domande"].sum(),
                       "utilizzabili": df["utilizzabili"].sum(),
                       "scartate": df["scartate"].sum(), "file_inutilizzabili": ""}
    return df, details


# estrazione frame
def extract_frame(video_path, sec, out_path):
    """Salva il frame all'istante `sec`. Prova decord, poi ripiega su OpenCV.

    Serve per image_nutrition_estimation, dove ogni domanda mostra 5 fotogrammi
    presi da 5 video diversi a timestamp indicati nelle annotation.
    """
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    try:
        from decord import VideoReader
        from PIL import Image
        vr = VideoReader(video_path, num_threads=1)
        i = min(int(round(sec * vr.get_avg_fps())), len(vr) - 1)
        Image.fromarray(vr[i].asnumpy()).save(out_path, quality=95)
        return out_path
    except Exception:
        pass
    import cv2
    cap = cv2.VideoCapture(video_path)
    try:
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        i = int(round(sec * fps))
        if total:
            i = min(i, total - 1)
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ok, frame = cap.read()
        if not ok:                                  # seek per indice fallito
            cap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000.0)
            ok, frame = cap.read()
        if not ok:
            raise RuntimeError(f"frame non leggibile a {sec:.3f}s in {video_path}")
        cv2.imwrite(out_path, frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
    finally:
        cap.release()
    return out_path


def _visual_input(item, task, index, panels_index, input_mode, prefer,
                  frame_extractor, frame_cache):
    """Prepara cio' che va passato al modello -> (input, etichetta sorgente)."""
    if TASK_FILES[task][1] == "images":
        paths = []
        for key in sorted(item["inputs"], key=lambda k: int(re.search(r"\d+", k).group())):
            inp = item["inputs"][key]
            video, _ = resolve_video(inp["id"], index, prefer="full")
            sec = hhmmss_to_sec(inp["time"])
            out = os.path.join(frame_cache, f"{inp['id']}_{int(round(sec*1000)):09d}.jpg")
            if not os.path.exists(out):
                (frame_extractor or extract_frame)(video, sec, out)
            paths.append(out)
        return paths, "frames"
    vid = item["inputs"]["video 1"]["id"]
    if input_mode == "panels":
        return panels_index[vid], "panels"
    path, src = resolve_video(vid, index, prefer=prefer)
    return path, src


# funzione centrale di inferenza
def run_task(task, ann_dir, index, infer_fn, out_jsonl, panels_index=None,
             input_mode="video", prefer="sampled", system=None,
             frame_extractor=None, frame_cache=None, limit=None,
             verbose=True, readable_cache=None):
    """Inferenza su un task, con ripresa da dove si era interrotto.

    Ogni risposta viene scritta su `out_jsonl` subito: se Colab si disconnette,
    rilanciare la cella riprende senza rifare le domande gia' fatte.
    """
    ann, skipped = usable_items(ann_dir, task, index, panels_index,
                                input_mode, prefer, readable_cache)
    if frame_cache:
        os.makedirs(frame_cache, exist_ok=True)
    if verbose:
        print(f"  {len(ann)} domande utilizzabili"
              + (f", {len(skipped)} escluse (file mancanti o corrotti)" if skipped else ""))

    done = {}
    if os.path.exists(out_jsonl):
        with open(out_jsonl) as f:
            for line in f:
                r = json.loads(line)
                done[r["key"]] = r
        if verbose and done:
            print(f"  ripresa: {len(done)} risposte gia' presenti")

    results = []
    for n, (key, item) in enumerate(ann.items()):
        if limit and n >= limit:
            break
        if key in done:
            results.append(done[key])
            continue

        question, choices, correct = item["question"], item["choices"], item["correct_idx"]
        rec = {"key": key, "task": task, "input_mode": input_mode,
               "q_type": q_type_of(task, question),
               "video_id": list(item["inputs"].values())[0]["id"],
               "n_choices": len(choices), "correct_idx": correct}
        try:
            visual, src = _visual_input(item, task, index, panels_index, input_mode,
                                        prefer, frame_extractor, frame_cache)
            response = infer_fn(visual, build_prompt(question, choices, system=system))
            idx, method = parse_answer(response, choices)
            rec.update(source=src, response=response, predicted_idx=idx,
                       parse_method=method, error=None)
        except Exception as e:
            rec.update(source=None, response=None, predicted_idx=-1,
                       parse_method="error", error=f"{type(e).__name__}: {e}")

        rec["is_correct"] = bool(rec["predicted_idx"] == correct)
        with open(out_jsonl, "a") as f:
            f.write(json.dumps(rec) + "\n")
        results.append(rec)

        if verbose:
            mark = "ERR" if rec["error"] else ("OK " if rec["is_correct"] else "KO ")
            shown = (rec["response"] or rec["error"] or "")[:55]
            print(f"  [{n+1}/{len(ann)}] {mark} {key.split('_')[-1]:>3} | "
                  f"pred={rec['predicted_idx']} gt={correct} | "
                  f"{rec['parse_method']:<18} | {shown!r}")

    df = pd.DataFrame(results)
    df.attrs["skipped"] = skipped
    return df


def run_all(tasks, ann_dir, index, infer_fn, out_dir, tag, **kw):
    """Gira piu' task e concatena i risultati."""
    os.makedirs(out_dir, exist_ok=True)
    out, skipped = [], {}
    for task in tasks:
        print(f"\n{'='*70}\n  TASK: {task}   (run: {tag})\n{'='*70}")
        df = run_task(task, ann_dir, index, infer_fn,
                      os.path.join(out_dir, f"{tag}_{task}.jsonl"), **kw)
        skipped[task] = df.attrs.get("skipped", {})
        out.append(df)
    allr = pd.concat(out, ignore_index=True)
    allr.attrs["skipped"] = skipped
    return allr


# ================================================================= metriche
def accuracy_report(df, label="", paper_ref=None, ann_dir=None, show_bias=True):
    """Report allineato al protocollo del paper.

    HD-EPIC Tab. 2 e' la MEDIA DELLE ACCURACY PER PROTOTIPO, non l'accuracy
    aggregata sulle domande: con prototipi da 50/50/100 i due numeri non
    coincidono. Qui vengono stampati entrambi, cosi' il confronto e' esplicito.
    """
    out = {}
    per_task = df.groupby("task")["is_correct"].agg(corrette="sum", valutate="count")
    per_task["accuracy_%"] = (per_task["corrette"] / per_task["valutate"] * 100).round(1)
    if ann_dir:
        per_task["su_totale"] = [len(load_annotations(ann_dir, t)) for t in per_task.index]
    out["per_task"] = per_task
    out["nutrition_paper"] = round(per_task["accuracy_%"].mean(), 1)
    out["nutrition_pooled"] = round(df["is_correct"].mean() * 100, 1)

    print("=" * 70)
    print(f"  {label}   (n = {len(df)} domande valutate)")
    print("=" * 70)
    print(per_task.to_string())
    print(f"\n  Nutrition, metrica del paper (media per prototipo) : {out['nutrition_paper']}%")
    print(f"  Nutrition, accuracy aggregata sulle domande        : {out['nutrition_pooled']}%")
    print(f"  Baseline random                                    : 20.0%")
    if paper_ref in PAPER_NUTRITION:
        ref = PAPER_NUTRITION[paper_ref]
        print(f"  Paper, {paper_ref:<28}: {ref}%   "
              f"(delta {out['nutrition_paper'] - ref:+.1f})")

    inval = df[df["predicted_idx"] < 0]
    print(f"\n  Risposte non interpretate o in errore: {len(inval)}/{len(df)} "
          f"({len(inval)/max(len(df),1)*100:.1f}%)")
    print("  Come e' stata letta la risposta:")
    print("   ", df["parse_method"].value_counts().to_dict())
    out["parse_methods"] = df["parse_method"].value_counts()

    by_type = df.groupby("q_type")["is_correct"].agg(corrette="sum", valutate="count")
    by_type["accuracy_%"] = (by_type["corrette"] / by_type["valutate"] * 100).round(1)
    print("\n  Accuracy per tipo di domanda:")
    print(by_type.to_string())
    out["per_q_type"] = by_type

    if show_bias:
        print("\n  Bias posizionale (% risposte su ciascuna lettera, vs ground truth):")
        bias = {}
        for task, g in df.groupby("task"):
            n = int(g["n_choices"].max())
            pred = g[g["predicted_idx"] >= 0]["predicted_idx"].value_counts(normalize=True)
            gt = g["correct_idx"].value_counts(normalize=True)
            row = (pd.DataFrame({"pred": pred, "gt": gt})
                     .reindex(range(n)).fillna(0) * 100).round(0).astype(int)
            bias[task] = row["pred"]
            fmt = lambda c: "[" + " ".join(f"{v:3d}" for v in row[c]) + "]"
            print(f"    {task:<28} pred(A-E)={fmt('pred')}  gt(A-E)={fmt('gt')}")
        out["bias"] = pd.DataFrame(bias)
    return out


def compare_runs(runs, label="Confronto"):
    """Confronto fra piu' run: {nome: DataFrame} -> tabella accuracy per task."""
    rows = {}
    for name, df in runs.items():
        acc = (df.groupby("task")["is_correct"].mean() * 100).round(1)
        acc["NUTRITION (media prototipi)"] = round(acc.mean(), 1)
        rows[name] = acc
    tab = pd.DataFrame(rows)
    if len(runs) == 2:
        a, b = list(runs)
        tab["delta"] = (tab[b] - tab[a]).round(1)
    print("=" * 70)
    print(f"  {label}")
    print("=" * 70)
    print(tab.to_string())
    return tab

Writing /content/hdepic_eval.py


## 1.2 Percorsi e indicizzazione dei file

I video sono sparsi in piu' cartelle (`videos_*_sampled`, `videos_*`, `videos_*_panels`) e
lo stesso video puo' servire a piu' prototipi pur essendo stato scaricato in una sola
cartella. Per questo si costruisce **un indice globale** `video_id -> path` invece di
cercare in una cartella fissa per task: cercando per cartella si perdono domande che
sarebbero perfettamente eseguibili.

In [12]:
import os, glob

BASE = "/content/drive/MyDrive/Progetto Deep Learning 1"
ANN_DIR    = BASE                      # dove stanno i 3 json di annotation
OUT_DIR    = os.path.join(BASE, "risultati")
FRAME_DIR  = os.path.join(BASE, "frames_image_nutrition_estimation")

# tutte le cartelle di video e di pannelli presenti nel progetto
VIDEO_ROOTS = sorted(d for d in glob.glob(os.path.join(BASE, "*"))
                     if os.path.isdir(d) and "panel" not in os.path.basename(d).lower()
                     and os.path.basename(d).lower().startswith(("video", "videos")))
PANEL_ROOTS = sorted(d for d in glob.glob(os.path.join(BASE, "*"))
                     if os.path.isdir(d) and "panel" in os.path.basename(d).lower())

print("cartelle video :", [os.path.basename(d) for d in VIDEO_ROOTS])
print("cartelle panels:", [os.path.basename(d) for d in PANEL_ROOTS])

VIDEO_INDEX = H.build_video_index(VIDEO_ROOTS)
PANEL_INDEX = H.build_panels_index(PANEL_ROOTS)
print(f"\nindicizzati: {len(VIDEO_INDEX['sampled'])} video campionati, "
      f"{len(VIDEO_INDEX['full'])} video integrali, {len(PANEL_INDEX)} cartelle di pannelli")

# controllo che le annotation ci siano davvero
for t, (fname, _) in H.TASK_FILES.items():
    p = os.path.join(ANN_DIR, fname)
    print(("  OK " if os.path.exists(p) else "  MANCA ") + fname)

cartelle video : ['Videos Nutrition Change Sampled', 'Videos Nutrition Estimation Sampled', 'videos_image_nutrition_estimation_sampled']
cartelle panels: ['videos_image_nutrition_estimation_panels', 'videos_nutrition_change_panels', 'videos_nutrition_estimation_panels']

indicizzati: 46 video campionati, 0 video integrali, 46 cartelle di pannelli
  OK nutrition_video_nutrition_estimation.json
  OK nutrition_nutrition_change.json
  OK nutrition_image_nutrition_estimation.json


## Quante domande sono davvero eseguibili

Nel dataset scaricato ci sono mp4 **da 0 byte** e mp4 **troncati**: esistono come file ma
non si aprono. Se non li si individua prima, ogni domanda che li usa produce un'eccezione
che finisce contata come risposta sbagliata, e l'accuracy risulta piu' bassa del vero.

Il pre-volo apre davvero ogni file e separa le domande eseguibili da quelle da escludere.
Le escluse **non entrano nel denominatore**: i risultati verranno riportati con il loro N.

In [13]:
READABLE = {}   # cache: aprire i video da Drive e' lento, si controlla una volta sola

cov, dettagli = H.preflight(ANN_DIR, VIDEO_INDEX, cache=READABLE)
print(cov.to_string(index=False))

rotti = sorted({v for task in dettagli.values() for ids in task.values() for v in ids})
if rotti:
    print(f"\nFile mancanti o non decodificabili ({len(rotti)}):")
    for v in rotti:
        print("   ", v)

                      task  input  domande  utilizzabili  scartate file_inutilizzabili
video_nutrition_estimation  video       50            46         4                   1
          nutrition_change  video       50            50         0                   0
image_nutrition_estimation frames      100            88        12                   4
                    TOTALE             200           184        16                    

File mancanti o non decodificabili (4):
    P06-20240510-090328
    P06-20240510-140459
    P07-20240529-131737
    P07-20240529-134410


## 1.4 Il modello

L'unica parte specifica di VideoLLaMA 2 e' la funzione `infer_fn`. Il `processor['video']`
del repo accetta sia il path di un mp4 sia una **lista di path di immagini** e in entrambi i
casi applica il preprocessing corretto per l'encoder CLIP (resize 336x336, normalizzazione
CLIP, padding `expand2square`). Costruire il tensore a mano con normalizzazioni sbagliate
produce output senza senso, quindi si passa sempre dal processor.

Da notare: il processor ricampiona a `model.config.num_frames` (8 di default, massimo 32).
Il campionamento a 128 frame fatto a monte **non** arriva quindi tutto al modello: serve solo
a rendere i file piu' leggeri e uniformi.

In [14]:
from videollama2 import mm_infer

MAX_NEW_TOKENS = 32          # basta: la risposta attesa e' una lettera
USE_SYSTEM_PROMPT = None     # il paper non usa Systemp Prompt (se lo utilizzassimo le performance si alzano)

def videollama2_infer(visual, prompt):
    """visual: path di un mp4, oppure lista di path di immagini (frame o pannelli)."""
    tensor = processor["video"](visual)
    return mm_infer(tensor, prompt, model, tokenizer, modal="video",
                    do_sample=False,          # greedy: run deterministico e ripetibile
                    max_new_tokens=MAX_NEW_TOKENS)

print("num_frames usati dal processor:", getattr(model.config, "num_frames", 8))
print("dtype:", next(model.parameters()).dtype)

num_frames usati dal processor: 8
dtype: torch.float16


## 1.6 Inferenza completa

I risultati vengono scritti su file **una domanda alla volta**: se Colab si disconnette,
basta rilanciare la cella e riparte da dove era arrivata invece di rifare tutto.

In [16]:
baseline = H.run_all(list(H.TASK_FILES), ANN_DIR, VIDEO_INDEX, videollama2_infer,
                     out_dir=OUT_DIR, tag="videollama2_baseline",
                     system=USE_SYSTEM_PROMPT,
                     frame_extractor=H.extract_frame, frame_cache=FRAME_DIR,
                     readable_cache=READABLE)

print(f"\nCompletate {len(baseline)} domande")


  TASK: video_nutrition_estimation   (run: videollama2_baseline)
  46 domande utilizzabili, 4 escluse (file mancanti o corrotti)


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:497: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


  [1/46] OK    0 | pred=3 gt=3 | leading_letter     | 'D'
  [2/46] KO    1 | pred=2 gt=0 | leading_letter     | 'C'
  [3/46] OK    2 | pred=4 gt=4 | leading_letter     | 'E'
  [4/46] OK    3 | pred=4 gt=4 | leading_letter     | 'E'
  [5/46] OK    4 | pred=1 gt=1 | leading_letter     | 'B'
  [6/46] OK    5 | pred=4 gt=4 | leading_letter     | 'E'
  [7/46] OK    6 | pred=2 gt=2 | leading_letter     | 'C'
  [8/46] OK    7 | pred=3 gt=3 | leading_letter     | 'D'
  [9/46] OK    8 | pred=3 gt=3 | leading_letter     | 'D'
  [10/46] KO    9 | pred=1 gt=4 | leading_letter     | 'B'
  [11/46] OK   10 | pred=2 gt=2 | leading_letter     | 'C'
  [12/46] OK   11 | pred=4 gt=4 | leading_letter     | 'E'
  [13/46] OK   12 | pred=1 gt=1 | leading_letter     | 'B'
  [14/46] KO   13 | pred=2 gt=0 | leading_letter     | 'C'
  [15/46] KO   14 | pred=4 gt=0 | leading_letter     | 'E'
  [16/46] OK   15 | pred=4 gt=4 | leading_letter     | 'E'
  [17/46] OK   16 | pred=0 gt=0 | leading_letter     | 'A'
  [18/

In [17]:
rep_base = H.accuracy_report(baseline,
                             label="VideoLLaMA 2 7B - baseline (video)",
                             paper_ref="VideoLLaMA 2 7B",
                             ann_dir=ANN_DIR)#costruisco il report

  VideoLLaMA 2 7B - baseline (video)   (n = 184 domande valutate)
                            corrette  valutate  accuracy_%  su_totale
task                                                                 
image_nutrition_estimation        21        88        23.9        100
nutrition_change                   8        50        16.0         50
video_nutrition_estimation        32        46        69.6         50

  Nutrition, metrica del paper (media per prototipo) : 36.5%
  Nutrition, accuracy aggregata sulle domande        : 33.2%
  Baseline random                                    : 20.0%
  Paper, VideoLLaMA 2 7B             : 32.7%   (delta +3.8)

  Risposte non interpretate o in errore: 0/184 (0.0%)
  Come e' stata letta la risposta:
    {'leading_letter': 184}

  Accuracy per tipo di domanda:
                  corrette  valutate  accuracy_%
q_type                                          
calories                10        37        27.0
carbs                   17        29      

### Come leggere il bias posizionale

L'ultima tabella confronta **su quale lettera il modello mette le sue risposte** con la
distribuzione reale delle risposte corrette. Se il modello fosse solo impreciso, le due
distribuzioni si assomiglierebbero. Se invece concentra le risposte su A e B mentre il
ground truth e' uniforme, allora non sta rispondendo alla domanda: sta seguendo una
preferenza a priori sul formato. E' un modo per distinguere *"il modello sbaglia"* da
*"il modello non sta guardando il video"*, ed e' la spiegazione tipica di un'accuracy
**sotto** il 20% del random.

---
# Parte 2 — Video Panels (Visaul Prompting)

Secondo esperimento, basato su *Video Panels for Long Video Understanding. L'idea: invece di dare al modello 8 fotogrammi sparsi, gli si
danno 8 **immagini pannellate 2x2**, ognuna contenente 4 fotogrammi consecutivi. A parita'
di finestra di contesto il modello copre cosi' 32 fotogrammi invece di 8, scambiando
risoluzione spaziale con risoluzione temporale.

I pannelli sono stati generati a monte da `paper_mosaic_sampling/mosaic.py` (32 pannelli per
video, griglia 2x2, quindi 128 fotogrammi coperti).

**Ambito del confronto:** solo i due prototipi basati su video. `image_nutrition_estimation`
resta fuori perche' il suo input sono 5 fotogrammi isolati da video diversi: non c'e' una
sequenza temporale da impacchettare, e il metodo del paper non si applica.

In [18]:
cov_p, dett_p = H.preflight(ANN_DIR, VIDEO_INDEX, tasks=H.VIDEO_TASKS,
                            panels_index=PANEL_INDEX, input_mode="panels",
                            cache=READABLE)
print(cov_p.to_string(index=False))

# quanti pannelli per video, e quanti ne vede davvero il modello
if PANEL_INDEX:
    esempio = next(iter(PANEL_INDEX.items()))
    print(f"\nesempio: {esempio[0]} -> {len(esempio[1])} pannelli")
    print(f"il processor ne campiona {getattr(model.config, 'num_frames', 8)}, "
          f"cioe' {getattr(model.config, 'num_frames', 8) * 4} fotogrammi originali "
          f"contro gli {getattr(model.config, 'num_frames', 8)} del baseline")

                      task  input  domande  utilizzabili  scartate file_inutilizzabili
video_nutrition_estimation panels       50            46         4                   1
          nutrition_change panels       50            50         0                   0
                    TOTALE             100            96         4                    

esempio: P01-20240202-161354 -> 32 pannelli
il processor ne campiona 8, cioe' 32 fotogrammi originali contro gli 8 del baseline


In [19]:
panels_run = H.run_all(H.VIDEO_TASKS, ANN_DIR, VIDEO_INDEX, videollama2_infer,
                       out_dir=OUT_DIR, tag="videollama2_panels",
                       panels_index=PANEL_INDEX, input_mode="panels",
                       system=USE_SYSTEM_PROMPT, readable_cache=READABLE)

rep_pan = H.accuracy_report(panels_run,
                            label="VideoLLaMA 2 7B - Video Panels 2x2",
                            ann_dir=ANN_DIR)


  TASK: video_nutrition_estimation   (run: videollama2_panels)
  46 domande utilizzabili, 4 escluse (file mancanti o corrotti)


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:497: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


  [1/46] OK    0 | pred=3 gt=3 | leading_letter     | 'D'
  [2/46] KO    1 | pred=2 gt=0 | leading_letter     | 'C'
  [3/46] OK    2 | pred=4 gt=4 | leading_letter     | 'E'
  [4/46] OK    3 | pred=4 gt=4 | leading_letter     | 'E'
  [5/46] OK    4 | pred=1 gt=1 | leading_letter     | 'B'
  [6/46] OK    5 | pred=4 gt=4 | leading_letter     | 'E'
  [7/46] OK    6 | pred=2 gt=2 | leading_letter     | 'C'
  [8/46] OK    7 | pred=3 gt=3 | leading_letter     | 'D'
  [9/46] OK    8 | pred=3 gt=3 | leading_letter     | 'D'
  [10/46] OK    9 | pred=4 gt=4 | leading_letter     | 'E'
  [11/46] OK   10 | pred=2 gt=2 | leading_letter     | 'C'
  [12/46] OK   11 | pred=4 gt=4 | leading_letter     | 'E'
  [13/46] OK   12 | pred=1 gt=1 | leading_letter     | 'B'
  [14/46] KO   13 | pred=2 gt=0 | leading_letter     | 'C'
  [15/46] KO   14 | pred=2 gt=0 | leading_letter     | 'C'
  [16/46] OK   15 | pred=4 gt=4 | leading_letter     | 'E'
  [17/46] KO   16 | pred=4 gt=0 | leading_letter     | 'E'
  [18/

## 2.1 Confronto

Il confronto va fatto **sulle stesse domande**: il baseline viene quindi ristretto ai due
prototipi video e alle domande effettivamente valutate anche nel run con i pannelli.

In [20]:
comuni = set(panels_run["key"]) & set(baseline["key"])
b = baseline[baseline["key"].isin(comuni)]
p = panels_run[panels_run["key"].isin(comuni)]
print(f"domande confrontate a parita' di input: {len(comuni)}\n")

tab = H.compare_runs({"baseline (video)": b, "panels 2x2": p},
                     "VideoLLaMA 2 7B: video liscio vs pannelli 2x2")

domande confrontate a parita' di input: 96

  VideoLLaMA 2 7B: video liscio vs pannelli 2x2
                             baseline (video)  panels 2x2  delta
task                                                            
nutrition_change                         16.0        20.0    4.0
video_nutrition_estimation               69.6        65.2   -4.4
NUTRITION (media prototipi)              42.8        42.6   -0.2


## 2.2 Salvataggio dei risultati

Tutto su CSV, una riga per domanda, con la risposta grezza del modello e il metodo con cui
e' stata interpretata: senza questi due campi un numero di accuracy non e' verificabile
da nessuno.

In [21]:
import pandas as pd

tutto = pd.concat([baseline.assign(run="baseline"),
                   panels_run.assign(run="panels")], ignore_index=True)
csv = os.path.join(OUT_DIR, "videollama2_risultati_completi.csv")
tutto.to_csv(csv, index=False)

riepilogo = pd.DataFrame({
    "run": ["baseline (video)", "panels 2x2"],
    "nutrition_media_prototipi_%": [rep_base["nutrition_paper"], rep_pan["nutrition_paper"]],
    "nutrition_aggregata_%":       [rep_base["nutrition_pooled"], rep_pan["nutrition_pooled"]],
    "domande_valutate":            [len(baseline), len(panels_run)],
})
riepilogo.to_csv(os.path.join(OUT_DIR, "videollama2_riepilogo.csv"), index=False)

print(riepilogo.to_string(index=False))
print(f"\nsalvato: {csv}")
print(f"paper, VideoLLaMA 2 7B su Nutrition: {H.PAPER_NUTRITION['VideoLLaMA 2 7B']}%  "
      f"| random: 20.0%")

             run  nutrition_media_prototipi_%  nutrition_aggregata_%  domande_valutate
baseline (video)                         36.5                   33.2               184
      panels 2x2                         42.6                   41.7                96

salvato: /content/drive/MyDrive/Progetto Deep Learning 1/risultati/videollama2_risultati_completi.csv
paper, VideoLLaMA 2 7B su Nutrition: 32.7%  | random: 20.0%
